# 00 — Colab setup and GPU benchmark (M0)

Sets up the FinCast-LSTM repo on Colab and measures LSTM training speed on this GPU.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 on the free tier).
2. In the Colab **Secrets** panel (key icon), add these and enable *Notebook access* for both:
   - `GH_TOKEN`: a fine-grained GitHub token with **read-only Contents** access to `moghbaie/FinCast-LSTM` (needed to clone the private repo).
   - `TIINGO_API_KEY`: your Tiingo token (used from M1 on).

Data, checkpoints and run outputs are stored in Google Drive under `MyDrive/FinCast-LSTM/`, so they survive disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FinCast-LSTM'
import os
for d in ['data', 'runs', 'results/benchmarks']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# Clone or update the private repo. The token is passed via an HTTP header for this call only:
# it is not printed and not stored in .git/config.
import base64, subprocess
from google.colab import userdata
REPO_DIR = '/content/FinCast-LSTM'
URL = 'https://github.com/moghbaie/FinCast-LSTM.git'
tok = userdata.get('GH_TOKEN')
hdr = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
auth = ['git', '-c', f'http.extraHeader={hdr}']
if os.path.exists(f'{REPO_DIR}/.git'):
    r = subprocess.run(auth + ['-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
else:
    r = subprocess.run(auth + ['clone', URL, REPO_DIR], capture_output=True, text=True)
print(r.returncode, r.stdout[-500:], r.stderr[-500:].replace(tok, '***'))
del tok, hdr
!git -C /content/FinCast-LSTM log --oneline -1

In [ ]:
# Keep Colab's preinstalled torch, install the package and its other dependencies.
!pip install -q -e "/content/FinCast-LSTM[data,models,viz,dev]"
# Link data/ and runs/ to Drive so outputs persist.
for d in ['data', 'runs']:
    link = f'{REPO_DIR}/{d}'
    if os.path.islink(link) or not os.path.exists(link):
        if os.path.islink(link): os.unlink(link)
        os.symlink(f'{DRIVE_ROOT}/{d}', link)
%cd /content/FinCast-LSTM

In [ ]:
# Gate: unit tests (CPU) + GPU test
!python -m pytest -W error -q
!python -m pytest -W error -q -m gpu

In [ ]:
# Benchmark (about 5 to 10 min on a T4). Saved to Drive.
import torch, re
gpu = re.sub(r'[^A-Za-z0-9]+', '_', torch.cuda.get_device_name(0)).strip('_').lower()
label = f'colab_{gpu}_v1'
!python scripts/benchmark_lstm.py --label {label} --out {DRIVE_ROOT}/results/benchmarks

**Send the result back:** copy the JSON lines printed above (or the file `MyDrive/FinCast-LSTM/results/benchmarks/colab_*_v1.json`) into the Claude Code chat. They are added to `results/benchmarks/m0_benchmark.md`.